In [ ]:
import os
import json
from PIL import Image
from perception_layer.multi_attribute_classifire.dataset_generation import CLIPPseudoLabeler, apply_category_prior
import matplotlib.pyplot as plt
import random

# Initialize the labeler (run your class cell first)
labeler = CLIPPseudoLabeler()

def visualize_prediction(img_path, ann, labeler):
    img = Image.open(img_path).convert("RGB")
    x, y, w, h = ann["bbox"]
    crop = img.crop((x, y, x + w, y + h))

    # Get labels
    labels = labeler.label(crop)

    # Apply priors (simulating your build_dataset logic)
    cat_id = ann["category_id"]
    weather_final = apply_category_prior(labels["weather"], cat_id, "weather")
    formality_final = apply_category_prior(labels["formality"], cat_id, "formality")

    # Display
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    ax[0].imshow(crop)
    ax[0].set_title(f"Category ID: {cat_id}")
    ax[0].axis('off')

    # Textual readout
    stats = (
        f"RAW SCORES:\n"
        f"Weather: {labels['weather']:.2f}\n"
        f"Formality: {labels['formality']:.2f}\n\n"
        f"AFTER PRIORS:\n"
        f"Weather: {weather_final:.2f}\n"
        f"Formality: {formality_final:.2f}\n\n"
        f"DISTRIBUTIONS:\n"
        f"Fit: {[f'{v:.2f}' for v in labels['fit'].tolist()]}\n"
        f"Style: {[f'{v:.2f}' for v in labels['style'].tolist()]}"
    )
    ax[1].text(0.1, 0.5, stats, fontsize=12, family='monospace', va='center')
    ax[1].axis('off')
    plt.show()

In [ ]:
# Test on a few random samples
with open('../../data/fashionpedia_coco/train/_annotations.coco.json') as f:
    coco = json.load(f)

# Pick 3 random samples to inspect
test_samples = random.sample(coco['annotations'], 20)
for ann in test_samples:
    img_file = next(i['file_name'] for i in coco['images'] if i['id'] == ann['image_id'])
    img_path = os.path.join('../../data/fashionpedia_coco/train', img_file)
    visualize_prediction(img_path, ann, labeler)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

def show_random_coco_sample(img_dir, ann_file):
    with open(ann_file) as f:
        coco = json.load(f)

    # Map category IDs to names for labeling
    cat_id_to_name = {cat['id']: cat['name'] for cat in coco['categories']}

    # Pick a random image that actually has annotations
    annotated_img_ids = list(set(ann['image_id'] for ann in coco['annotations']))
    img_id = random.choice(annotated_img_ids)

    # Get image info and its annotations
    img_info = next(img for img in coco['images'] if img['id'] == img_id)
    img_anns = [ann for ann in coco['annotations'] if ann['image_id'] == img_id]

    # Load image
    img_path = os.path.join(img_dir, img_info['file_name'])
    img = Image.open(img_path)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)

    # Draw boxes
    for ann in img_anns:
        x, y, w, h = ann['bbox']
        cat_name = cat_id_to_name.get(ann['category_id'], "Unknown")

        # Create a Rectangle patch
        rect = patches.Rectangle((x, y-5), w, h, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

        # Add label text
        plt.text(x, y, cat_name, color='white', fontsize=10,
                 bbox=dict(facecolor='red', alpha=0.5))

    plt.title(f"Image ID: {img_id} | {img_info['file_name']}")
    plt.axis('off')
    plt.show()

# Usage:
show_random_coco_sample('../../data/fashionpedia_coco/train', '../../data/fashionpedia_coco/train/_annotations.coco.json')

In [ ]:
import json

# Path to your newly generated training annotations
ann_file = '../../data/fashionpedia_coco/train/_annotations.coco.json'

with open(ann_file) as f:
    data = json.load(f)

# Look at the first annotation
first_ann = data['annotations'][0]
print(f"Sample BBox: {first_ann['bbox']}")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from datasets import load_dataset
from PIL import Image

def test_inference_and_verify(limit=5):
    print(f"--- Running Dry Run for {limit} images ---")
    full_ds = load_dataset("detection-datasets/fashionpedia", split="train")

    # Process only the first few samples
    test_samples = full_ds.select(range(limit))

    CLASS_NAMES = ['shirt, blouse', 'top, t-shirt, sweatshirt', 'sweater', 'cardigan', 'jacket', 'vest', 'pants', 'shorts', 'skirt', 'coat', 'dress', 'jumpsuit', 'cape', 'glasses', 'hat', 'headband, head covering, hair accessory', 'tie', 'glove', 'watch', 'belt', 'leg warmer', 'tights, stockings', 'sock', 'shoe', 'bag, wallet', 'scarf', 'umbrella', 'hood', 'collar', 'lapel', 'epaulette', 'sleeve', 'pocket', 'neckline', 'buckle', 'zipper', 'applique', 'bead', 'bow', 'flower', 'fringe', 'ribbon', 'rivet', 'ruffle', 'sequin', 'tassel']

    for idx, sample in enumerate(test_samples):
        img_w, img_h = sample['image'].size
        obj_data = sample['objects']

        fig, ax = plt.subplots(1, figsize=(8, 8))
        ax.imshow(sample['image'])

        print(f"\nImage {idx}: {img_w}x{img_h}")

        for i in range(len(obj_data['category'])):
            bbox = obj_data['bbox'][i]

            # --- YOUR CURRENT LOGIC CHECK ---
            # Fashionpedia (HF) provides [xmin, ymin, xmax, ymax] in absolute pixels.
            xmin = float(bbox[0])
            ymin = float(bbox[1])
            xmax = float(bbox[2])
            ymax = float(bbox[3])

            # COCO Format: [xmin, ymin, width, height]
            w = xmax - xmin
            h = ymax - ymin

            cat_name = CLASS_NAMES[obj_data['category'][i]] if obj_data['category'][i] < len(CLASS_NAMES) else "Unknown"

            # Draw standard Rectangle
            rect = patches.Rectangle((xmin, ymin), w, h, linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            plt.text(xmin, ymin - 5, cat_name, color='white', weight='bold', bbox=dict(facecolor='green', alpha=0.5))

            print(f"  - Detected: {cat_name} at [{xmin:.1f}, {ymin:.1f}, {w:.1f}, {h:.1f}]")

        plt.title(f"Test Sample {idx}")
        plt.axis('off')
        plt.show()

test_inference_and_verify()